# Main79 + Exp11 — Direct Stacking (FINAL)

**Run this notebook directly with `Run all`.**

### Before running
Upload **one ZIP** through the **Colab Files sidebar**. Do **not** use an upload cell.

The ZIP must contain:
- `train.json`
- `test.json`
- `main66.py`
- `main79_95_kaggle.py`
- `main64_oof_meta_features.npy`
- `main64_val_meta_features.npy`
- `main64_oof_svm.npy`
- `main64_oof_nbsvm.npy`
- `main64_oof_hgb.npy`
- `main64_oof_local.npy`
- `phase2_exp10_exp11_exp12.ipynb`

The notebook automatically finds the ZIP in `/content`, extracts it, verifies every required file, and then runs the complete validation → stacking → full-data test pipeline.

**No standalone Main79 notebook is required.**


In [1]:
# ============================================================
# 0. COLAB SIDEBAR / LOCAL ZIP CHECK + EXTRACTION
# ============================================================
# IMPORTANT:
# - Google Colab: Upload main79_stack_bundle.zip to /content before pressing "Run all".
# - Local / Jupyter: Keep main79_stack_bundle.zip in the notebook folder.
# No files.upload() is used anywhere in this notebook.

from pathlib import Path
import os, json, zipfile, shutil, random, time, importlib.util, sys
import numpy as np
import pandas as pd

required_names = {
    "train.json",
    "test.json",
    "main66.py",
    "main79_95_kaggle.py",
    "main64_oof_meta_features.npy",
    "main64_val_meta_features.npy",
    "main64_oof_svm.npy",
    "main64_oof_nbsvm.npy",
    "main64_oof_hgb.npy",
    "main64_oof_local.npy",
    "phase2_exp10_exp11_exp12.ipynb",
}

# Auto-detect directory containing the bundle ZIP:
# Supports Google Colab (/content), local execution (current notebook dir), and common subpaths
search_dirs = [
    Path.cwd(),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Stacking - main79 + exp11"),
    Path("/content"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/best_model_so_far"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1"),
]

selected_zip = None
CONTENT = None

for d in search_dirs:
    if not d.exists():
        continue
    for zp in sorted(d.glob("*.zip")):
        try:
            with zipfile.ZipFile(zp, "r") as z:
                names = {Path(n).name for n in z.namelist() if not n.endswith("/")}
            if required_names.issubset(names):
                selected_zip = zp
                CONTENT = d
                break
        except zipfile.BadZipFile:
            continue
    if selected_zip is not None:
        break

if selected_zip is None:
    checked_str = "\n".join(f"  - {str(d)}" for d in search_dirs if d.exists())
    raise FileNotFoundError(
        f"No valid ZIP bundle containing all required files was found.\n\n"
        f"Checked directories:\n{checked_str}\n\n"
        f"Expected a ZIP containing:\n" +
        "\n".join("  - " + x for x in sorted(required_names)) +
        "\n\nPlease ensure 'main79_stack_bundle.zip' is in the notebook directory "
        "or uploaded to /content on Colab."
    )

ROOT = CONTENT / "main79_stack_runtime"
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

with zipfile.ZipFile(selected_zip, "r") as z:
    z.extractall(ROOT)

all_files = {
    p.name: p
    for p in ROOT.rglob("*")
    if p.is_file()
}

missing = sorted(required_names - set(all_files))
if missing:
    raise RuntimeError(
        "Extraction completed but required files are missing:\n" +
        "\n".join("  - " + x for x in missing)
    )

# Put all required files in one flat directory so every later cell has
# deterministic paths.
WORK = ROOT / "files"
WORK.mkdir()

PATHS = {}
for name in sorted(required_names):
    dst = WORK / name
    shutil.copy2(all_files[name], dst)
    PATHS[name] = dst

print("=" * 72)
print("ZIP:", selected_zip.name)
print("LOCATION:", CONTENT)
print("FILES VERIFIED")
print("=" * 72)
for name in sorted(required_names):
    print("OK:", name)

ZIP: main79_stack_bundle.zip
LOCATION: /home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Stacking - main79 + exp11
FILES VERIFIED
OK: main64_oof_hgb.npy
OK: main64_oof_local.npy
OK: main64_oof_meta_features.npy
OK: main64_oof_nbsvm.npy
OK: main64_oof_svm.npy
OK: main64_val_meta_features.npy
OK: main66.py
OK: main79_95_kaggle.py
OK: phase2_exp10_exp11_exp12.ipynb
OK: test.json
OK: train.json


In [2]:
# ============================================================
# 1. IMPORTS + DATA + CANONICAL SPLIT
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

SEED = 42

def load_jsonl(path, labelled=True):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    ids = [r["id"] for r in rows]
    texts = [r["text"] for r in rows]

    if labelled:
        labels = np.asarray(
            [0 if r["label"] == "A" else 1 for r in rows],
            dtype=np.int64
        )
        return ids, texts, labels

    return ids, texts

train_ids, train_texts, labels = load_jsonl(
    PATHS["train.json"], True
)
test_ids, test_texts = load_jsonl(
    PATHS["test.json"], False
)

idx = np.arange(len(labels))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)

y_train = labels[train_idx]
y_val = labels[val_idx]

print("Documents:", len(labels))
print("A:", int(np.sum(labels == 0)))
print("B:", int(np.sum(labels == 1)))
print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_texts))

assert len(train_idx) == 8428
assert len(val_idx) == 2108
assert len(test_texts) == 3000


PyTorch: 2.13.0+cu130
CUDA: True
Documents: 10536
A: 3699
B: 6837
Training: 8428
Validation: 2108
Test: 3000


## 2. Main79 validation

This uses the **actual uploaded `main79_95_kaggle.py` architecture/configuration**, but trains the residual models on the canonical 8428/2108 split so we can measure Main79 before stacking.

The uploaded Main79 script itself is then used later for the final full-data test prediction.


In [3]:
# ============================================================
# 2. LOAD MAIN79 + MAIN66 AS NORMAL MODULES
# ============================================================
# No source-string exec. No giant embedded Python strings.

def import_module_from_file(module_name, path):
    spec = importlib.util.spec_from_file_location(
        module_name,
        str(path)
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

main79 = import_module_from_file(
    "main79_uploaded",
    PATHS["main79_95_kaggle.py"]
)

main66 = import_module_from_file(
    "main66_uploaded",
    PATHS["main66.py"]
)

print("Main79 imported.")
print("Main66 imported.")

# Exact Main79 constants, read from the uploaded script.
for name in [
    "MODEL_SEEDS", "BATCH_SIZE", "MAX_LEN", "EPOCHS",
    "LR", "WEIGHT_DECAY", "EMB_DIM", "HIDDEN",
    "DROPOUT", "RESIDUAL_WEIGHT", "META_C"
]:
    assert hasattr(main79, name), f"Missing Main79 constant: {name}"

print("Main79 constants verified.")


Main79 imported.
Main66 imported.
Main79 constants verified.


In [4]:
# ============================================================
# 3. MAIN79 LEAKAGE-SAFE TEACHER + VALIDATION DATA
# ============================================================
oof_meta = np.load(
    PATHS["main64_oof_meta_features.npy"]
).astype(np.float32)

val_meta = np.load(
    PATHS["main64_val_meta_features.npy"]
).astype(np.float32)

assert oof_meta.shape == (8428, 13)
assert val_meta.shape == (2108, 13)

meta_model = LogisticRegression(
    C=main79.META_C,
    max_iter=5000,
    solver="lbfgs",
    random_state=main79.SEED
)

meta_model.fit(
    oof_meta,
    labels[train_idx]
)

teacher_train_raw = meta_model.decision_function(
    oof_meta
).astype(np.float32)

teacher_val_raw = meta_model.decision_function(
    val_meta
).astype(np.float32)

teacher_scale = np.percentile(
    np.abs(teacher_train_raw),
    95
)

if teacher_scale < 1e-6:
    teacher_scale = 1.0

# Exact Main79 convention: teacher = raw / scale, clipped to [-6, 6].
teacher_train = np.clip(
    teacher_train_raw / teacher_scale,
    -6.0,
    6.0
).astype(np.float32)

teacher_val = np.clip(
    teacher_val_raw / teacher_scale,
    -6.0,
    6.0
).astype(np.float32)

meta_mean = np.mean(oof_meta, axis=0)
meta_std = np.std(oof_meta, axis=0)
meta_std = np.where(meta_std < 1e-6, 1.0, meta_std)

meta_train_z = np.clip(
    (oof_meta - meta_mean) / meta_std,
    -6.0,
    6.0
).astype(np.float32)

meta_val_z = np.clip(
    (val_meta - meta_mean) / meta_std,
    -6.0,
    6.0
).astype(np.float32)

teacher_only_acc = accuracy_score(
    y_val,
    teacher_val_raw >= 0
)

print("Teacher scale:", teacher_scale)
print("Teacher-only validation accuracy:", teacher_only_acc)
print("OOF meta:", oof_meta.shape)
print("Val meta:", val_meta.shape)

Teacher scale: 10.450376
Teacher-only validation accuracy: 0.9283681214421252
OOF meta: (8428, 13)
Val meta: (2108, 13)


In [5]:
# ============================================================
# 4. MAIN79 VALIDATION RESIDUAL ENSEMBLE
# ============================================================
# Use the exact Main79 ResidualDataset / ResidualBiGRU classes
# from the uploaded script.

main79.seed_everything(main79.SEED)

# Main79 expects integer token sequences.
for s in train_texts[:10]:
    assert isinstance(s, (list, tuple)), type(s)

VOCAB_SIZE = (
    max(int(t) for seq in train_texts for t in seq) + 1
)
PAD_IDX = VOCAB_SIZE

main79.VOCAB_SIZE = VOCAB_SIZE
main79.PAD_IDX = PAD_IDX
main79.DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
main79.USE_AMP = (
    main79.DEVICE.type == "cuda"
)

print("Device:", main79.DEVICE)
print("VOCAB_SIZE:", VOCAB_SIZE)
print("PAD_IDX:", PAD_IDX)

# Exact Main79 dataset class.
train_ds = main79.ResidualDataset(
    [train_texts[i] for i in train_idx],
    y_train,
    meta_train_z
)

val_ds = main79.ResidualDataset(
    [train_texts[i] for i in val_idx],
    y_val,
    meta_val_z
)

train_loader = DataLoader(
    train_ds,
    batch_size=main79.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(main79.DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=main79.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(main79.DEVICE.type == "cuda")
)

def predict_validation_residual(model):
    model.eval()
    out = []

    with torch.no_grad():
        for ids, mask, y, aux in val_loader:
            ids = ids.to(main79.DEVICE)
            mask = mask.to(main79.DEVICE)
            aux = aux.to(main79.DEVICE)

            residual = model(ids, mask, aux)
            out.append(residual.detach().float().cpu().numpy())

    return np.concatenate(out)

def train_main79_validation_seed(seed):
    print("\n" + "=" * 72)
    print("MAIN79 VALIDATION SEED", seed)
    print("=" * 72)

    main79.seed_everything(seed)

    model = main79.ResidualBiGRU(
        meta_train_z.shape[1]
    ).to(main79.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=main79.LR,
        weight_decay=main79.WEIGHT_DECAY
    )

    criterion = nn.BCEWithLogitsLoss()

    TRAIN_WEIGHT = 0.075

    for epoch in range(1, main79.EPOCHS + 1):
        model.train()
        losses = []

        for ids, mask, y, aux in train_loader:
            ids = ids.to(main79.DEVICE)
            mask = mask.to(main79.DEVICE)
            y = y.to(main79.DEVICE)
            aux = aux.to(main79.DEVICE)

            optimizer.zero_grad(set_to_none=True)

            residual = model(ids, mask, aux)
            teacher = aux[:, 0]

            final_logit = (
                teacher +
                TRAIN_WEIGHT * residual
            )

            loss = criterion(final_logit, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                main79.GRAD_CLIP
            )

            optimizer.step()
            losses.append(float(loss.item()))

        print(
            f"epoch {epoch:02d}/{main79.EPOCHS} "
            f"loss={np.mean(losses):.6f}"
        )

    residual = predict_validation_residual(model)

    # Main79 final validation/test-time residual shrinkage.
    score = (
        teacher_val +
        main79.RESIDUAL_WEIGHT * residual
    )

    return score

validation_scores = []

for seed in main79.MODEL_SEEDS:
    validation_scores.append(
        train_main79_validation_seed(seed)
    )

main79_val_score = np.mean(
    np.vstack(validation_scores),
    axis=0
).astype(np.float32)

main79_val_prob = 1.0 / (
    1.0 + np.exp(-main79_val_score)
)

main79_val_acc = accuracy_score(
    y_val,
    main79_val_score >= 0
)

main79_val_auc = roc_auc_score(
    y_val,
    main79_val_prob
)

print("\n" + "=" * 72)
print("MAIN79 VALIDATION RESULT")
print("=" * 72)
print("Accuracy:", main79_val_acc)
print("AUC:", main79_val_auc)


Device: cuda
VOCAB_SIZE: 18438
PAD_IDX: 18438

MAIN79 VALIDATION SEED 42
epoch 01/6 loss=0.468745
epoch 02/6 loss=0.299223
epoch 03/6 loss=0.232100
epoch 04/6 loss=0.213491
epoch 05/6 loss=0.207935
epoch 06/6 loss=0.197031

MAIN79 VALIDATION SEED 123
epoch 01/6 loss=0.455199
epoch 02/6 loss=0.283829
epoch 03/6 loss=0.231562
epoch 04/6 loss=0.213904
epoch 05/6 loss=0.205322
epoch 06/6 loss=0.196503

MAIN79 VALIDATION SEED 777
epoch 01/6 loss=0.459239
epoch 02/6 loss=0.288662
epoch 03/6 loss=0.226849
epoch 04/6 loss=0.213962
epoch 05/6 loss=0.204055
epoch 06/6 loss=0.200752

MAIN79 VALIDATION RESULT
Accuracy: 0.9369070208728653
AUC: 0.9803530504188399


## 3. Exp11 validation + full-data test

This section reproduces the **actual Exp11 TF-IDF + Structural architecture** from the uploaded Phase-2 notebook.

It does not execute the old notebook blindly. That is intentional: the old notebook's setup expected `/content/data/train.json`, which caused the earlier `FileNotFoundError`.

Here we use the already-verified ZIP paths directly.

The validation stage uses the same 80/20 split and the same Exp11 architecture/hyperparameters. After validation, the selected epoch is used to retrain Exp11 on all 10,536 labelled documents and predict the 3,000 test documents.


In [6]:
# ============================================================
# 5. EXP11 — EXACT HELPERS / ARCHITECTURE
# ============================================================
# These definitions are copied from the uploaded Exp11 notebook,
# with no experimental parameter changes.

EXP11_SEED = 42
EXP11_MAX_LEN = 384
EXP11_EMB_DIM = 128
EXP11_CNN_CHANNELS = 128
EXP11_Z_DIM = 128
EXP11_BATCH_SIZE = 64
EXP11_EPOCHS = 15
EXP11_PATIENCE = 3
EXP11_LR = 2e-3
EXP11_WEIGHT_DECAY = 1e-4
EXP11_TFIDF_MAX_FEATURES = 200_000
EXP11_TFIDF_MIN_DF = 3
EXP11_SVD_DIM = 256

def entropy_from_counts(counts):
    c = np.asarray(counts, dtype=float)
    c = c[c > 0]
    if len(c) == 0:
        return 0.0
    p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def sequence_features(doc):
    x = list(doc)
    n = len(x)
    if n == 0:
        return np.zeros(58, dtype=np.float32)

    cnt = __import__("collections").Counter(x)
    bigrams = list(zip(x[:-1], x[1:]))
    trigrams = list(zip(x[:-2], x[1:-1], x[2:]))

    def div(seq):
        return len(set(seq)) / max(1, len(seq))

    def rep(seq):
        return 1.0 - div(seq)

    half = max(1, n // 2)
    q = max(1, n // 4)

    first = x[:half]
    second = x[half:]
    q1, q2, q3, q4 = x[:q], x[q:2*q], x[2*q:3*q], x[3*q:]

    feats = [
        n, len(cnt), len(cnt)/n,
        len([v for v in cnt.values() if v > 1])/n,
        max(cnt.values())/n,
        np.mean(list(cnt.values())),
        np.std(list(cnt.values())),
        entropy_from_counts(list(cnt.values())),
        len(bigrams), len(set(bigrams)), div(bigrams), rep(bigrams),
        len(trigrams), len(set(trigrams)), div(trigrams), rep(trigrams),
        sum(x[i] == x[i-1] for i in range(1,n))/max(1,n-1),
        len(set(zip(x[:-1], x[1:])))/max(1,n-1),
        entropy_from_counts(list(__import__("collections").Counter(bigrams).values())),
        entropy_from_counts(list(__import__("collections").Counter(trigrams).values())),
        len(set(first))/len(first),
        len(set(second))/max(1,len(second)),
        entropy_from_counts(list(__import__("collections").Counter(first).values())),
        entropy_from_counts(list(__import__("collections").Counter(second).values())),
        len(set(q1))/len(q1), len(set(q2))/len(q2),
        len(set(q3))/len(q3), len(set(q4))/max(1,len(q4)),
        entropy_from_counts(list(__import__("collections").Counter(q1).values())),
        entropy_from_counts(list(__import__("collections").Counter(q2).values())),
        entropy_from_counts(list(__import__("collections").Counter(q3).values())),
        entropy_from_counts(list(__import__("collections").Counter(q4).values())),
        len(first), len(second), len(q1), len(q2), len(q3), len(q4),
        len(set(first)&set(second))/max(1,len(set(first)|set(second))),
        len(set(q1)&set(q2))/max(1,len(set(q1)|set(q2))),
        len(set(q2)&set(q3))/max(1,len(set(q2)|set(q3))),
        len(set(q3)&set(q4))/max(1,len(set(q3)|set(q4))),
        np.mean([len(t) for t in x]),
        np.std([len(t) for t in x]),
        min([len(t) for t in x]),
        max([len(t) for t in x]),
        np.mean([len(t) for t in x[:half]]),
        np.mean([len(t) for t in x[half:]]) if second else 0.0,
        len(x[0]), len(x[-1]),
        len(set(x[:min(10,n)]))/min(10,n),
        len(set(x[max(0,n-10):]))/min(10,n),
        entropy_from_counts(list(__import__("collections").Counter(x[:min(10,n)]).values())),
        entropy_from_counts(list(__import__("collections").Counter(x[max(0,n-10):]).values())),
        len(set(x[:min(25,n)]))/min(25,n),
        len(set(x[max(0,n-25):]))/min(25,n),
        entropy_from_counts(list(__import__("collections").Counter(x[:min(25,n)]).values())),
        entropy_from_counts(list(__import__("collections").Counter(x[max(0,n-25):]).values())),
    ]

    return np.asarray(feats, dtype=np.float32)

def build_vocab(all_docs):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for doc in all_docs:
        for t in doc:
            if t not in vocab:
                vocab[t] = len(vocab)
    return vocab

def encode_docs(docs, vocab):
    out = []
    for doc in docs:
        ids = [vocab.get(t, 1) for t in doc[:EXP11_MAX_LEN]]
        ids += [0] * (EXP11_MAX_LEN - len(ids))
        out.append(ids)
    return np.asarray(out, dtype=np.int64)

class TokenDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = sequences
        self.y = np.asarray(labels, dtype=np.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return (
            torch.tensor(self.x[i], dtype=torch.long),
            torch.tensor(self.y[i])
        )

class TokenEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(
            vocab_size, EXP11_EMB_DIM, padding_idx=0
        )
        self.convs = nn.ModuleList([
            nn.Conv1d(
                EXP11_EMB_DIM,
                EXP11_CNN_CHANNELS,
                k,
                padding=k//2
            )
            for k in (3,5,7)
        ])
        self.proj = nn.Sequential(
            nn.Linear(EXP11_CNN_CHANNELS * 3, EXP11_Z_DIM),
            nn.GELU(),
            nn.LayerNorm(EXP11_Z_DIM),
        )
    def forward(self, x):
        h = self.emb(x).transpose(1,2)
        pooled = []
        for conv in self.convs:
            z = torch.nn.functional.gelu(conv(h))
            z = torch.max(z, dim=-1).values
            pooled.append(z)
        return self.proj(torch.cat(pooled, dim=1))

class MLPEncoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256,EXP11_Z_DIM),
            nn.GELU(),
            nn.LayerNorm(EXP11_Z_DIM),
        )
    def forward(self,x):
        return self.net(x)

class FusionModel(nn.Module):
    def __init__(self, vocab_size, dims):
        super().__init__()
        self.token = TokenEncoder(vocab_size)
        self.branches = nn.ModuleDict()
        for name,d in dims.items():
            self.branches[name] = MLPEncoder(d)
        n = 128 * (1 + len(dims))
        self.fusion = nn.Sequential(
            nn.Linear(n,256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(256,EXP11_Z_DIM),
            nn.GELU(),
            nn.LayerNorm(EXP11_Z_DIM),
        )
        self.head = nn.Linear(EXP11_Z_DIM,1)

    def forward(self,tok,branch_inputs):
        zs = [self.token(tok)]
        for name in self.branches:
            zs.append(self.branches[name](branch_inputs[name]))
        z = self.fusion(torch.cat(zs,dim=1))
        return self.head(z).squeeze(1), z

def make_exp11_structural(docs):
    return np.vstack([sequence_features(d) for d in docs]).astype(np.float32)

def make_exp11_loaders(tok_train, ytr, branches_train,
                       tok_val, yv, branches_val):
    class MultiInputDataset(Dataset):
        def __init__(self,tok,y,b):
            self.tok=tok
            self.y=np.asarray(y,dtype=np.float32)
            self.b=b
        def __len__(self): return len(self.y)
        def __getitem__(self,i):
            return (
                torch.tensor(self.tok[i],dtype=torch.long),
                torch.tensor(self.y[i],dtype=torch.float32),
                {k:torch.tensor(v[i],dtype=torch.float32)
                 for k,v in self.b.items()}
            )

    tr = MultiInputDataset(tok_train,ytr,branches_train)
    va = MultiInputDataset(tok_val,yv,branches_val)

    return (
        DataLoader(tr,batch_size=EXP11_BATCH_SIZE,shuffle=True),
        DataLoader(va,batch_size=EXP11_BATCH_SIZE,shuffle=False)
    )


In [7]:
# ============================================================
# 6. EXP11 VALIDATION
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def run_exp11_validation():
    # Original Exp11 convention: A=1, B=0.
    exp_y_train = (labels[train_idx] == 0).astype(np.float32)
    exp_y_val = (labels[val_idx] == 0).astype(np.float32)

    docs_tr = [
        [str(t) for t in train_texts[i]]
        for i in train_idx
    ]
    docs_va = [
        [str(t) for t in train_texts[i]]
        for i in val_idx
    ]

    vocab = build_vocab(docs_tr)
    tok_tr = encode_docs(docs_tr, vocab)
    tok_va = encode_docs(docs_va, vocab)

    strings_tr = [" ".join(d) for d in docs_tr]
    strings_va = [" ".join(d) for d in docs_va]

    vec = TfidfVectorizer(
        ngram_range=(1,6),
        min_df=EXP11_TFIDF_MIN_DF,
        max_features=EXP11_TFIDF_MAX_FEATURES,
        sublinear_tf=True,
        token_pattern=r"(?u)\S+",
        dtype=np.float32
    )

    A = vec.fit_transform(strings_tr)
    B = vec.transform(strings_va)

    ncomp = min(EXP11_SVD_DIM, A.shape[1]-1)
    svd = TruncatedSVD(
        n_components=ncomp,
        random_state=EXP11_SEED
    )

    A_svd = svd.fit_transform(A).astype(np.float32)
    B_svd = svd.transform(B).astype(np.float32)

    Xraw = np.vstack([
        make_exp11_structural(docs_tr),
        make_exp11_structural(docs_va)
    ])

    scaler = StandardScaler()
    Xscaled = scaler.fit_transform(Xraw).astype(np.float32)

    branches_tr = {
        "tfidf": A_svd,
        "structural": Xscaled[:len(docs_tr)]
    }
    branches_va = {
        "tfidf": B_svd,
        "structural": Xscaled[len(docs_tr):]
    }

    tr_loader, va_loader = make_exp11_loaders(
        tok_tr, exp_y_train, branches_tr,
        tok_va, exp_y_val, branches_va
    )

    dims = {
        k: v.shape[1]
        for k,v in branches_tr.items()
    }

    model = FusionModel(len(vocab), dims).to(
        torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

    device = next(model.parameters()).device

    pos_weight = float(
        (exp_y_train == 0).sum() /
        max(1, (exp_y_train == 1).sum())
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight],
            device=device
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=EXP11_LR,
        weight_decay=EXP11_WEIGHT_DECAY
    )

    best_auc = -np.inf
    best_state = None
    best_epoch = 0
    bad = 0

    for epoch in range(1, EXP11_EPOCHS+1):
        model.train()
        for tok,y,b in tr_loader:
            tok=tok.to(device)
            y=y.to(device)
            b={k:v.to(device) for k,v in b.items()}

            optimizer.zero_grad(set_to_none=True)
            logits,_=model(tok,b)
            loss=criterion(logits,y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()

        model.eval()
        probs=[]
        ys=[]

        with torch.no_grad():
            for tok,y,b in va_loader:
                tok=tok.to(device)
                b={k:v.to(device) for k,v in b.items()}
                logits,_=model(tok,b)
                probs.extend(
                    torch.sigmoid(logits).cpu().numpy()
                )
                ys.extend(y.numpy())

        auc=roc_auc_score(ys,probs)
        print(f"Exp11 epoch {epoch:02d} | val AUC {auc:.5f}")

        if auc > best_auc:
            best_auc=auc
            best_epoch=epoch
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            bad=0
        else:
            bad += 1
            if bad >= EXP11_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()

    p=[]
    with torch.no_grad():
        for tok,y,b in va_loader:
            tok=tok.to(device)
            b={k:v.to(device) for k,v in b.items()}
            logits,_=model(tok,b)
            p.extend(
                torch.sigmoid(logits).cpu().numpy()
            )

    pA=np.asarray(p,dtype=np.float32)

    # Convert A-positive -> B-positive.
    pB=1.0-pA

    acc=accuracy_score(y_val,pB>=0.5)
    auc=roc_auc_score(y_val,pB)

    return {
        "model": model,
        "best_epoch": best_epoch,
        "val_prob_A": pA,
        "val_prob_B": pB,
        "val_acc": float(acc),
        "val_auc": float(auc),
        "vocab": vocab,
    }

exp11_val = run_exp11_validation()

print("\n" + "="*72)
print("EXP11 VALIDATION RESULT")
print("="*72)
print("Accuracy:", exp11_val["val_acc"])
print("AUC:", exp11_val["val_auc"])
print("Best epoch:", exp11_val["best_epoch"])


Exp11 epoch 01 | val AUC 0.94851
Exp11 epoch 02 | val AUC 0.95842
Exp11 epoch 03 | val AUC 0.96185
Exp11 epoch 04 | val AUC 0.96024
Exp11 epoch 05 | val AUC 0.96396
Exp11 epoch 06 | val AUC 0.96257
Exp11 epoch 07 | val AUC 0.96462
Exp11 epoch 08 | val AUC 0.96071
Exp11 epoch 09 | val AUC 0.96380
Exp11 epoch 10 | val AUC 0.96474
Exp11 epoch 11 | val AUC 0.96331
Exp11 epoch 12 | val AUC 0.96211
Exp11 epoch 13 | val AUC 0.96258

EXP11 VALIDATION RESULT
Accuracy: 0.8937381404174574
AUC: 0.9647359530583215
Best epoch: 10


In [8]:
# ============================================================
# 7. VALIDATION ERROR OVERLAP + BLEND
# ============================================================
main79_pred = main79_val_prob >= 0.5
exp11_pred = exp11_val["val_prob_B"] >= 0.5
truth = y_val == 1

print("Main79 accuracy:", main79_val_acc)
print("Main79 AUC:", main79_val_auc)
print("Exp11 accuracy:", exp11_val["val_acc"])
print("Exp11 AUC:", exp11_val["val_auc"])

print("\nERROR OVERLAP")
print("Both correct:",
      int(np.sum((main79_pred == truth) & (exp11_pred == truth))))
print("Main79 only:",
      int(np.sum((main79_pred == truth) & (exp11_pred != truth))))
print("Exp11 only:",
      int(np.sum((main79_pred != truth) & (exp11_pred == truth))))
print("Both wrong:",
      int(np.sum((main79_pred != truth) & (exp11_pred != truth))))

best_blend = None

for alpha in np.linspace(0,1,101):
    p = (
        alpha * main79_val_prob +
        (1-alpha) * exp11_val["val_prob_B"]
    )

    acc = accuracy_score(y_val,p>=0.5)
    auc = roc_auc_score(y_val,p)

    if best_blend is None or acc > best_blend["accuracy"]:
        best_blend = {
            "alpha":float(alpha),
            "accuracy":float(acc),
            "auc":float(auc)
        }

print("\nBest transparent blend:")
print(best_blend)


Main79 accuracy: 0.9369070208728653
Main79 AUC: 0.9803530504188399
Exp11 accuracy: 0.8937381404174574
Exp11 AUC: 0.9647359530583215

ERROR OVERLAP
Both correct: 1829
Main79 only: 146
Exp11 only: 55
Both wrong: 78

Best transparent blend:
{'alpha': 0.89, 'accuracy': 0.9392789373814042, 'auc': 0.9822279516358463}


## 4. Full-data Main79 test prediction

Now the uploaded **original Main79 script** is run as a normal Python process using the verified files. This is the exact final Main79 implementation, not a retyped approximation.


In [ ]:
# ============================================================
# 8. RUN ORIGINAL MAIN79 FULL-DATA TEST PIPELINE
# ============================================================
MAIN79_RUNTIME = CONTENT / "main79_full_runtime"

if MAIN79_RUNTIME.exists():
    shutil.rmtree(MAIN79_RUNTIME)

MAIN79_RUNTIME.mkdir()

for name in [
    "train.json",
    "test.json",
    "main66.py",
    "main64_oof_meta_features.npy",
    "main64_val_meta_features.npy",
    "main64_oof_svm.npy",
    "main64_oof_nbsvm.npy",
    "main64_oof_hgb.npy",
    "main64_oof_local.npy",
]:
    shutil.copy2(PATHS[name], MAIN79_RUNTIME / name)

shutil.copy2(
    PATHS["main79_95_kaggle.py"],
    MAIN79_RUNTIME / "main79_95_kaggle.py"
)

print("Running uploaded Main79 script...")
print("This is the original full-data Main79 implementation.")

result = __import__("subprocess").run(
    [sys.executable, "main79_95_kaggle.py"],
    cwd=str(MAIN79_RUNTIME),
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Original Main79 script failed with exit code {result.returncode}."
    )

score_path = (
    MAIN79_RUNTIME /
    "main79_outputs" /
    "test_final_score.npy"
)

if not score_path.exists():
    raise FileNotFoundError(
        "Original Main79 finished, but main79_outputs/test_final_score.npy "
        "was not created."
    )

main79_test_score = np.load(score_path).astype(np.float32)
main79_test_prob = 1.0 / (
    1.0 + np.exp(-main79_test_score)
)

assert main79_test_prob.shape == (3000,)
assert np.isfinite(main79_test_prob).all()

print("Main79 test probabilities:", main79_test_prob.shape)


Running uploaded Main79 script...
This is the original full-data Main79 implementation.


In [ ]:
# ============================================================
# 9. EXP11 FULL-DATA RETRAIN + TEST PREDICTION
# ============================================================
# Same Exp11 architecture/hyperparameters.
# Use the validation-selected epoch count.
# Fit all 10,536 labelled rows, then transform/predict all 3,000 test rows.

best_epoch = int(exp11_val["best_epoch"])

all_docs = [
    [str(t) for t in s]
    for s in train_texts
]

test_docs = [
    [str(t) for t in s]
    for s in test_texts
]

full_vocab = build_vocab(all_docs)

tok_all = encode_docs(all_docs, full_vocab)
tok_test = encode_docs(test_docs, full_vocab)

all_strings = [" ".join(d) for d in all_docs]
test_strings = [" ".join(d) for d in test_docs]

vec_full = TfidfVectorizer(
    ngram_range=(1,6),
    min_df=EXP11_TFIDF_MIN_DF,
    max_features=EXP11_TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    token_pattern=r"(?u)\S+",
    dtype=np.float32
)

A = vec_full.fit_transform(all_strings)
T = vec_full.transform(test_strings)

ncomp = min(EXP11_SVD_DIM, A.shape[1]-1)

svd_full = TruncatedSVD(
    n_components=ncomp,
    random_state=EXP11_SEED
)

A_svd = svd_full.fit_transform(A).astype(np.float32)
T_svd = svd_full.transform(T).astype(np.float32)

struct_all = make_exp11_structural(all_docs)
struct_test = make_exp11_structural(test_docs)

struct_scaler = StandardScaler()
struct_all = struct_scaler.fit_transform(
    struct_all
).astype(np.float32)

struct_test = struct_scaler.transform(
    struct_test
).astype(np.float32)

branches_all = {
    "tfidf": A_svd,
    "structural": struct_all
}

branches_test = {
    "tfidf": T_svd,
    "structural": struct_test
}

class TestDataset(Dataset):
    def __init__(self,tok,b):
        self.tok=tok
        self.b=b
    def __len__(self):
        return len(self.tok)
    def __getitem__(self,i):
        return (
            torch.tensor(self.tok[i],dtype=torch.long),
            {k:torch.tensor(v[i],dtype=torch.float32)
             for k,v in self.b.items()}
        )

full_y_A = (labels == 0).astype(np.float32)

class FullExpDataset(Dataset):
    def __init__(self,tok,y,b):
        self.tok=tok
        self.y=np.asarray(y,dtype=np.float32)
        self.b=b
    def __len__(self):
        return len(self.y)
    def __getitem__(self,i):
        return (
            torch.tensor(self.tok[i],dtype=torch.long),
            torch.tensor(self.y[i],dtype=torch.float32),
            {k:torch.tensor(v[i],dtype=torch.float32)
             for k,v in self.b.items()}
        )

full_ds = FullExpDataset(
    tok_all, full_y_A, branches_all
)

full_loader = DataLoader(
    full_ds,
    batch_size=EXP11_BATCH_SIZE,
    shuffle=True
)

test_ds = TestDataset(
    tok_test,
    branches_test
)

test_loader = DataLoader(
    test_ds,
    batch_size=EXP11_BATCH_SIZE,
    shuffle=False
)

dims_full = {
    k:v.shape[1]
    for k,v in branches_all.items()
}

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

full_model = FusionModel(
    len(full_vocab),
    dims_full
).to(device)

pos_weight = float(
    (full_y_A == 0).sum() /
    max(1,(full_y_A == 1).sum())
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [pos_weight],
        device=device
    )
)

optimizer = torch.optim.AdamW(
    full_model.parameters(),
    lr=EXP11_LR,
    weight_decay=EXP11_WEIGHT_DECAY
)

print("Exp11 full-data training epochs:", best_epoch)

for epoch in range(1,best_epoch+1):
    full_model.train()
    losses=[]

    for tok,y,b in full_loader:
        tok=tok.to(device)
        y=y.to(device)
        b={k:v.to(device) for k,v in b.items()}

        optimizer.zero_grad(set_to_none=True)
        logits,_=full_model(tok,b)
        loss=criterion(logits,y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            full_model.parameters(),1.0
        )
        optimizer.step()
        losses.append(float(loss.item()))

    print(
        f"Exp11 full epoch {epoch:02d}/{best_epoch} "
        f"loss={np.mean(losses):.6f}"
    )

full_model.eval()
test_prob_A=[]

with torch.no_grad():
    for tok,b in test_loader:
        tok=tok.to(device)
        b={k:v.to(device) for k,v in b.items()}
        logits,_=full_model(tok,b)
        test_prob_A.extend(
            torch.sigmoid(logits).cpu().numpy()
        )

exp11_test_prob_A=np.asarray(
    test_prob_A,
    dtype=np.float32
)

exp11_test_prob_B=1.0-exp11_test_prob_A

assert exp11_test_prob_B.shape == (3000,)
assert np.isfinite(exp11_test_prob_B).all()

print("Exp11 test probabilities:", exp11_test_prob_B.shape)


## 5. Final candidate submissions

The validation blend weight is selected from the transparent 101-point grid.

No Kaggle submission is made automatically.


In [ ]:
# ============================================================
# 10. WRITE + VERIFY CANDIDATE SUBMISSIONS
# ============================================================
alpha = best_blend["alpha"]

blend_test_prob = (
    alpha * main79_test_prob +
    (1-alpha) * exp11_test_prob_B
)

def write_submission(filename, prob):
    prob = np.asarray(prob).reshape(-1)

    assert len(prob) == 3000
    assert np.isfinite(prob).all()

    pred = np.where(prob >= 0.5, "B", "A")

    sub = pd.DataFrame({
        "id": test_ids,
        "label": pred
    })

    assert len(sub) == 3000
    assert sub["id"].nunique() == 3000
    assert list(sub.columns) == ["id","label"]
    assert set(sub["label"].unique()).issubset({"A","B"})

    path = CONTENT / filename
    sub.to_csv(path,index=False)

    print(
        f"CREATED {filename} | "
        f"rows={len(sub)} | "
        f"A={int(np.sum(pred=='A'))} | "
        f"B={int(np.sum(pred=='B'))}"
    )

    return path

submission_main79 = write_submission(
    "submission_main79_reproduced.csv",
    main79_test_prob
)

submission_exp11 = write_submission(
    "submission_exp11.csv",
    exp11_test_prob_B
)

submission_blend = write_submission(
    "submission_main79_exp11_blend.csv",
    blend_test_prob
)

print("\n" + "="*72)
print("DONE")
print("="*72)
print("Main79 validation accuracy:", main79_val_acc)
print("Main79 validation AUC:", main79_val_auc)
print("Exp11 validation accuracy:", exp11_val["val_acc"])
print("Exp11 validation AUC:", exp11_val["val_auc"])
print("Blend alpha:", alpha)
print("Blend validation accuracy:", best_blend["accuracy"])
print("Blend validation AUC:", best_blend["auc"])
print("\nFiles:")
print(submission_main79)
print(submission_exp11)
print(submission_blend)
